# metrics.ipynb

Fungsi loss dan metrik untuk segmentasi biner manusia.

**Objek pelatihan (TensorFlow murni, dijalankan setiap batch):** `dice_coef`, `dice_loss`, `focal_loss`, `combined_loss` (objektif), dan metrik `iou` keras.

**Khusus evaluasi (NumPy/SciPy):** `boundary_iou` (alias `boundary_accuracy`).

> Objek pelatihan ditulis hanya dengan operasi TF native. Versi sebelumnya membungkus IoU dalam `tf.numpy_function`, yang memaksa round trip CPU setiap batch dan memecah `model.compile()` pada Keras 3 ("Cannot take the length of shape with unknown rank"). Versi TF murni di bawah ini menghindari kedua masalah tersebut.

Notebook lain memuat definisi ini dengan `%run metrics.ipynb`.

In [ ]:
import numpy as np
import tensorflow as tf

SMOOTH = 1e-6

In [ ]:
try:
    _DEMO
except NameError:
    _DEMO = True  # True saat notebook ini dijalankan sendiri; disetel False oleh pemanggil %run

### Koefisien Dice dan Dice loss

Overlap soft yang terdiferensiasi. Robust terhadap ketidakseimbangan foreground/background yang berat karena mengabaikan area background besar yang mudah.

In [ ]:
def dice_coef(y_true, y_pred):
    """Koefisien Dice soft pada seluruh batch (terdiferensiasi)."""
    y_true = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_pred = tf.reshape(tf.cast(y_pred, tf.float32), [-1])
    intersection = tf.reduce_sum(y_true * y_pred)
    return (2.0 * intersection + SMOOTH) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + SMOOTH
    )


def dice_loss(y_true, y_pred):
    """1 - Dice. Robust terhadap ketidakseimbangan foreground/background."""
    return 1.0 - dice_coef(y_true, y_pred)

### Focal loss

Cross-entropy yang diskalakan oleh `(1 - p)^gamma`, yang menurunkan bobot piksel yang sudah terklasifikasi dengan baik sehingga gradien terkonsentrasi pada piksel sulit dan piksel batas.

In [ ]:
def focal_loss(y_true, y_pred, gamma=2.0, alpha=0.25):
    """
    Focal loss biner. Menurunkan bobot piksel yang sudah terklasifikasi
    dengan baik sehingga gradien terkonsentrasi pada piksel sulit / batas.

    gamma : kekuatan fokus (lebih tinggi -> lebih banyak penekanan piksel mudah)
    alpha : bobot yang diberikan ke kelas foreground
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1.0 - 1e-7)
    cross_entropy = -(
        y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred)
    )
    weight = (
        y_true * alpha * tf.pow(1.0 - y_pred, gamma)
        + (1.0 - y_true) * (1.0 - alpha) * tf.pow(y_pred, gamma)
    )
    return tf.reduce_mean(weight * cross_entropy)

### Loss kombinasi (objektif pelatihan)

`dice_loss + focal_loss`.

In [ ]:
def combined_loss(y_true, y_pred):
    """Objektif pelatihan: Dice loss + Focal loss."""
    return dice_loss(y_true, y_pred) + focal_loss(y_true, y_pred)

### IoU (metrik pelatihan)

Jaccard keras setelah thresholding pada 0.5, dihitung per sampel lalu dirata-ratakan. TF murni dengan rank yang diketahui, sehingga bisa dikompilasi sebagai metrik pelatihan.

In [ ]:
def iou(y_true, y_pred, threshold=0.5):
    """
    IoU keras (Jaccard) dihitung per sampel lalu dirata-ratakan pada batch.

    TF murni, rank-4 NHWC. Prediksi di-threshold pada `threshold`, sehingga
    ini mencerminkan IoU yang akan Anda dapatkan dari mask biner akhir,
    bukan skor soft.
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(tf.cast(y_pred, tf.float32) > threshold, tf.float32)
    axes = [1, 2, 3]
    intersection = tf.reduce_sum(y_true * y_pred, axis=axes)
    union = (
        tf.reduce_sum(y_true, axis=axes)
        + tf.reduce_sum(y_pred, axis=axes)
        - intersection
    )
    return tf.reduce_mean((intersection + SMOOTH) / (union + SMOOTH))

### Boundary IoU (khusus evaluasi)

IoU yang dibatasi pada pita batas mask ground-truth. NumPy/SciPy, dimaksudkan untuk dipanggil sekali per gambar dalam evaluasi, bukan sebagai metrik pelatihan yang dikompilasi.

In [ ]:
def boundary_iou(y_true, y_pred, dilation=2):
    """
    IoU yang dibatasi pada pita batas mask ground-truth.

    Pita batas = dilasi(mask) - erosi(mask). Skor tinggi berarti tepi
    prediksi berada dekat dengan tepi sebenarnya. NumPy + SciPy, dimaksudkan
    untuk dipanggil sekali per gambar di eval.py, bukan sebagai metrik
    pelatihan yang dikompilasi.

    Menerima array 2-D (H, W) biner (0/1 atau 0/255). Mengembalikan float.
    """
    from scipy.ndimage import binary_dilation, binary_erosion

    y_true_bin = (np.asarray(y_true) > 0.5).astype(np.uint8)
    y_pred_bin = (np.asarray(y_pred) > 0.5).astype(np.uint8)

    dilated = binary_dilation(y_true_bin, iterations=dilation)
    eroded = binary_erosion(y_true_bin, iterations=dilation)
    boundary = np.clip(dilated.astype(np.int8) - eroded.astype(np.int8), 0, 1).astype(bool)

    if boundary.sum() == 0:
        return 1.0  # degenerasi (mask kosong / penuh); tidak ada yang bisa dinilai di tepi

    true_b = y_true_bin[boundary]
    pred_b = y_pred_bin[boundary]
    intersection = float((true_b * pred_b).sum())
    union = float(true_b.sum() + pred_b.sum() - intersection)
    if union == 0:
        return 1.0
    return (intersection + 1e-15) / (union + 1e-15)


# Alias kompatibilitas mundur yang digunakan oleh kode evaluasi lama.
boundary_accuracy = boundary_iou

### Self-check

Uji kewarasan kecil (tidak perlu GPU atau model).

In [ ]:
if _DEMO:
    # Self-check kecil (tidak perlu GPU / model).
    yt = tf.constant(np.zeros((2, 8, 8, 1), np.float32))
    yp = tf.constant(np.zeros((2, 8, 8, 1), np.float32))
    yt = tf.tensor_scatter_nd_update(
        yt, [[0, i, j, 0] for i in range(2, 6) for j in range(2, 6)], [1.0] * 16
    )
    yp = tf.tensor_scatter_nd_update(
        yp, [[0, i, j, 0] for i in range(3, 6) for j in range(3, 6)], [1.0] * 9
    )
    print("dice_coef   :", float(dice_coef(yt, yp)))
    print("dice_loss   :", float(dice_loss(yt, yp)))
    print("focal_loss  :", float(focal_loss(yt, yp)))
    print("kombinasi   :", float(combined_loss(yt, yp)))
    print("iou         :", float(iou(yt, yp)))
    m_t = yt.numpy()[0, :, :, 0]
    m_p = yp.numpy()[0, :, :, 0]
    print("boundary_iou:", boundary_iou(m_t, m_p))